# FNO-family A1/A2: does a published architecture learn the 3D-ICE die-edge artefact? (report 9.32)

Same test as `scripts/artefact_learning.py` (ThermoNO), on plain FNO and CNO-FNO(+attention) from `src/fno/`.
Training protocol = `scripts/fno_fair_cv.py` (rise target, detrended loss, early stopping held off until 200 epochs, OneCycle, seed 0);
folds = `kfold_indices(n, 5, 0)` over the name-sorted scenarios, identical to artefact_learning.py. Geometry5 and geometry6.

* **A1** model trained on the PRE-snap data (dataset `thermo-3dic-shelf-v5-presnap`), scored on its own held-out pre-snap scenarios.
* **A2** the same trained model applied unchanged to the corrected (grid-snapped) versions of the same held-out scenarios (dataset `thermo-3dic-shelf-v5`).
* Control (plain FNO only): `fno_trained_new`, a second FNO trained on the corrected data, scored on the same held-out corrected scenarios.

Output: `/kaggle/working/fno_artefact/geometry{5,6}_seed0.json` in the same {summary, rows} shape as results/artefact_learning/*.json
(columns `truth_old`, `truth_new`, `<variant>_old`, `<variant>_on_new`, `fno_trained_new`). Resumable: finished folds are skipped on restart.
Attach both datasets. Internet on, GPU T4.

In [ ]:
import subprocess, sys, shutil, torch
from pathlib import Path
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pyamg'], check=True)
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
CLONE = Path('/kaggle/working/repo')
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print('repo at', subprocess.run(['git', '-C', str(CLONE), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())

# stage the two datasets: files under a path containing 'presnap' are the OLD data, the rest are the corrected data
STAGE = Path('/kaggle/working/stage')
shutil.rmtree(STAGE, ignore_errors=True)
for g in ['geometry5', 'geometry6']:
    for tag in ('old', 'new'):
        (STAGE / tag / g).mkdir(parents=True)
    files = sorted(Path('/kaggle/input').rglob(g + '_*.npz'))
    for p in files:
        tag = 'old' if 'presnap' in str(p).lower() else 'new'
        (STAGE / tag / g / p.name).symlink_to(p)
    n_old, n_new = len(list((STAGE / 'old' / g).iterdir())), len(list((STAGE / 'new' / g).iterdir()))
    print(g, 'old', n_old, 'new', n_new)
    assert n_old == 45 and n_new == 45, 'expected 45 scenarios in each of the pre-snap and corrected datasets'


In [ ]:
# A1/A2 artefact-learning test on the FNO family (mirrors scripts/artefact_learning.py for ThermoNO; report 9.32).
# Training = scripts/fno_fair_cv.py (rise target, detrended loss, early stopping held off until min_epochs, OneCycle).
# Folds = scripts/artefact_learning.py (kfold_indices(n, 5, seed 0) over the name-sorted scenarios).
import json, sys, time
from pathlib import Path
import numpy as np
import torch


def setup(repo):
    sys.path.insert(0, str(repo))
    global get_geometry_by_name, points_to_grid, FNODataset, build_lt_fno, build_cno_fno, build_fno
    global compute_norm_stats, hotspot_metrics, kfold_indices, per_scenario_stats
    from src.core.geometry_builders import get_geometry_by_name
    from src.core.mesh import points_to_grid
    from src.fno.data_loader import FNODataset
    from src.fno.ltfno import build_lt_fno
    from src.fno.model import build_cno_fno, build_fno
    from src.pinn.data_loader import compute_norm_stats
    from scripts.hotspot_eval import hotspot_metrics, kfold_indices   # metric code used by artefact_learning.py
    from scripts.layout_cv import per_scenario_stats


# ---- copied verbatim from scripts/artefact_learning.py ------------------------------------------------------------
def artefact(p, P, true_argmax=None):
    i = int(p.argmax())
    out = {'argmax_unpowered': bool(P[i] <= 0), 'excess_K': float(p.max() - p[P > 0].max())}
    if true_argmax is not None:
        out['same_node_as_truth'] = bool(i == true_argmax)
    return out


def summarise(rows, keys):
    s = {}
    for k in keys:
        v = [r[k] for r in rows if r.get(k) is not None]
        s[k] = float(np.mean(v)) if k not in ('hotspot_loc_err_um',) else float(np.median(v))
    return s
# ---- copied verbatim from scripts/fno_fair_cv.py (build, stack, loss_fn) -------------------------------------------
def build(variant, grid, a, dev):
    if variant == 'fno':
        return build_fno(grid, modes=tuple(a['modes']), hidden_ch=a['ch'], n_blocks=a['blocks'], device=dev)
    if variant == 'lt-fno':
        return build_lt_fno(grid, modes=tuple(a['modes']), hidden_ch=a['ch'], n_blocks=a['blocks'], device=dev)
    return build_cno_fno(grid, ch=a['ch'], n_fno_blocks=a['blocks'], n_cno_layers=2, use_attention=True, n_heads=4, device=dev)


def stack(ds, files, norm, dev):
    """Inputs as batched tensors, and the rise target (K) on the model grid."""
    it = ds.items
    t_rng = norm.T_max - norm.T_min
    amb = np.array([float(np.load(f, allow_pickle=True)['metadata'].item().get('t_ambient_kelvin',
                    float(np.load(f, allow_pickle=True)['metadata'].item()['t_ambient_celsius']) + 273.15))
                    for f in files], dtype=np.float32)
    T = torch.stack([x['T_norm'] for x in it]) * t_rng + norm.T_min
    rise = T - torch.as_tensor(amb)[:, None, None, None]
    X = [torch.stack([x[k] for x in it]).to(dev) for k in ('Q_norm', 'layer_id_norm', 'htc_norm', 't_amb_norm', 'tsv_frac')]
    return X, rise.to(dev), torch.as_tensor(amb, device=dev)


def loss_fn(out, tgt):
    """MSE on the rise plus MSE on the detrended rise, so spatial structure is not swamped by the offset."""
    d = lambda z: z - z.mean(dim=(1, 2, 3), keepdim=True)
    return ((out - tgt) ** 2).mean() + ((d(out) - d(tgt)) ** 2).mean()
# --------------------------------------------------------------------------------------------------------------------


def train(variant, grid, geom, fit_f, val_f, a, dev, seed):
    """Same loop as fno_fair_cv.run_fold; returns the best-val model, norm stats, target scale."""
    norm = compute_norm_stats(fit_f, {geom: get_geometry_by_name(geom)})
    Xf, Rf, _ = stack(FNODataset(fit_f, norm, grid), fit_f, norm, dev)
    Xv, Rv, _ = stack(FNODataset(val_f, norm, grid), val_f, norm, dev)
    scale = float(Rf.std())
    torch.manual_seed(seed)
    model = build(variant, grid, a, dev)
    steps = a['epochs'] * int(np.ceil(len(fit_f) / a['batch']))
    opt = torch.optim.AdamW(model.parameters(), lr=a['lr'], weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, a['lr'], total_steps=steps)
    best, best_state, stale, t0 = np.inf, None, 0, time.time()
    fwd = lambda X, idx: model(*[x[idx] for x in X])
    for ep in range(a['epochs']):
        model.train()
        for b in torch.randperm(len(fit_f), device=dev).split(a['batch']):
            opt.zero_grad()
            loss_fn(fwd(Xf, b), Rf[b] / scale).backward()
            opt.step()
            sched.step()
        model.eval()
        with torch.no_grad():
            v = float(np.mean([loss_fn(fwd(Xv, torch.tensor([i], device=dev)), Rv[i:i + 1] / scale).item()
                               for i in range(len(val_f))]))
        if v < best - 1e-7:
            best, best_state, stale = v, {k: t.detach().clone() for k, t in model.state_dict().items()}, 0
        else:
            stale += 1
        if stale >= a['patience'] and ep + 1 >= a['min_epochs']:
            break
    model.load_state_dict(best_state)
    model.eval()
    return model, norm, scale, ep + 1, time.time() - t0


def predict(model, norm, scale, files, grid, dev):
    """Flat (grid-order) prediction and truth, in K, for each file; plus grid-order power and coords."""
    X, R, A = stack(FNODataset(files, norm, grid), files, norm, dev)
    out = []
    with torch.no_grad():
        for j, f in enumerate(files):
            o = model(*[x[torch.tensor([j], device=dev)] for x in X])[0] * scale
            pred = (o + A[j]).cpu().numpy().ravel().astype(np.float64)
            true = (R[j] + A[j]).cpu().numpy().ravel().astype(np.float64)
            d = np.load(f, allow_pickle=True)
            c = d['coords'].astype(np.float64)
            cg = np.stack(points_to_grid(c, c[:, 0], c[:, 1], c[:, 2]), -1).reshape(-1, 3)
            P = points_to_grid(c, d['power'].astype(np.float64)).ravel()
            out.append((pred, true, P, cg))
    return out


def list_files(root, g):
    return sorted(Path(root).rglob(f'{g}_*.npz'), key=lambda p: p.name)


def run(old_root, new_root, out_dir, geoms, variants, folds, a, dev, train_new_variants=('fno',), max_train=None):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    keys = ['argmax_unpowered', 'excess_K', 'same_node_as_truth', 'r2', 'det_mae', 'hotspot_loc_err_um',
            'top1pct_recall', 'abs_peak_temp_err_K']
    for g in geoms:
        fo, fn = list_files(old_root, g), list_files(new_root, g)
        assert [p.name for p in fo] == [p.name for p in fn], 'pre/post-snap scenario lists differ'
        n = len(fo)
        c0 = np.load(fo[0])['coords']
        grid = tuple(len(np.unique(c0[:, i])) for i in (1, 0, 2))
        print(f'{g}: {n} scenarios, grid {grid}', flush=True)
        part = out_dir / f'{g}_seed0.partial.json'
        state = json.loads(part.read_text()) if part.exists() else {}     # key "variant|fold" -> {rows, epochs, seconds}
        for variant in variants:
            for fi0, te in enumerate(kfold_indices(n, 5, 0)):
                if fi0 not in folds or f'{variant}|{fi0}' in state:
                    continue
                fi = fi0 + 1                                    # fno_fair_cv numbers folds from 1 for its val split
                tr = np.setdiff1d(np.arange(n), te)
                if max_train:
                    tr = tr[:max_train]
                rng = np.random.default_rng(0 + fi)
                val_idx = set(rng.choice(len(tr), size=max(3, len(tr) // 9), replace=False).tolist())
                fit_o = [fo[i] for k, i in enumerate(tr) if k not in val_idx]
                val_o = [fo[i] for k, i in enumerate(tr) if k in val_idx]
                te_o, te_n = [fo[i] for i in te], [fn[i] for i in te]
                t = time.time()
                model, norm, scale, eps, secs = train(variant, grid, g, fit_o, val_o, a, dev, 0)
                p_old = predict(model, norm, scale, te_o, grid, dev)                   # A1
                p_new = predict(model, norm, scale, te_n, grid, dev)                   # A2 (training normalisation kept)
                del model
                p_ctl = None
                if variant in train_new_variants:                                      # control: trained on corrected data
                    fit_n = [fn[i] for k, i in enumerate(tr) if k not in val_idx]
                    val_n = [fn[i] for k, i in enumerate(tr) if k in val_idx]
                    m2, nrm2, sc2, eps2, _ = train(variant, grid, g, fit_n, val_n, a, dev, 0)
                    p_ctl = predict(m2, nrm2, sc2, te_n, grid, dev)
                    del m2
                fold_rows = {}
                for j, i in enumerate(te):
                    po, yo, Po, co = p_old[j]
                    pn, yn, Pn, cn = p_new[j]
                    ta = int(yo.argmax())
                    r = {'truth_old': artefact(yo, Po), 'truth_new': artefact(yn, Pn),
                         f'{variant}_old': {**artefact(po, Po, ta), **per_scenario_stats(yo, po), **hotspot_metrics(po, yo, co)},
                         f'{variant}_on_new': {**artefact(pn, Pn), **per_scenario_stats(yn, pn), **hotspot_metrics(pn, yn, cn)}}
                    if p_ctl is not None:
                        pc = p_ctl[j][0]
                        r[f'{variant}_trained_new'] = {**artefact(pc, Pn), **per_scenario_stats(yn, pc), **hotspot_metrics(pc, yn, cn)}
                    fold_rows[fo[i].stem] = r
                state[f'{variant}|{fi0}'] = {'fold': fi0, 'rows': fold_rows, 'epochs': eps, 'seconds': secs}
                part.write_text(json.dumps(state, default=float))
                write_final(state, g, out_dir, keys)
                if dev.type == 'cuda':
                    torch.cuda.empty_cache()
                print(f'{g} {variant} fold {fi0}: {eps} epochs ({secs / 60:.1f} min train), total {(time.time() - t) / 60:.1f} min', flush=True)


def write_final(state, g, out_dir, keys):
    merged = {}
    for k, v in state.items():
        for scen, cols in v['rows'].items():
            m = merged.setdefault(scen, {'scenario': scen, 'fold': v['fold']})
            m.update(cols)
    rows = [merged[s] for s in sorted(merged)]
    summary = {'n': len(rows)}
    cols = []
    for r in rows:
        for c in r:
            if c not in ('scenario', 'fold') and c not in cols:
                cols.append(c)
    for col in cols:
        sub = [r[col] for r in rows if col in r]
        summary[col] = summarise(sub, keys)
        summary[col]['n'] = len(sub)
    summary['train_epochs'] = {k: v['epochs'] for k, v in state.items()}
    (Path(out_dir) / f'{g}_seed0.json').write_text(json.dumps({'summary': summary, 'rows': rows}, indent=1, default=float))


In [ ]:
setup(CLONE)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# fno_fair_cv.py settings as used by notebooks/kaggle_fno_fair_a.ipynb: --epochs 300 --min-epochs 200 --patience 40, batch 2, lr 1e-3, ch 32, 4 blocks, modes 16 16 8
A = dict(ch=32, blocks=4, modes=[16, 16, 8], epochs=300, min_epochs=200, patience=40, batch=2, lr=1e-3)
OUT = Path('/kaggle/working/fno_artefact')
run(STAGE / 'old', STAGE / 'new', OUT, ['geometry5', 'geometry6'], ['fno', 'cno-fno-attn'], [0, 1, 2, 3, 4], A, dev,
    train_new_variants=('fno',))

In [ ]:
import json
for g in ['geometry5', 'geometry6']:
    s = json.loads((OUT / f'{g}_seed0.json').read_text())['summary']
    print(g, 'n =', s['n'])
    for col in [c for c in s if c not in ('n', 'train_epochs')]:
        v = s[col]
        print(f"  {col:<22} argmax_unpowered {v['argmax_unpowered']:.3f}  excess_K {v['excess_K']:.3f}  same_node {v['same_node_as_truth']:.3f}  "
              f"r2 {v['r2']:.3f}  det_mae {v['det_mae']:.3f}  loc_um {v['hotspot_loc_err_um']:.0f}  recall {v['top1pct_recall']:.3f}  |peak| {v['abs_peak_temp_err_K']:.2f}")